# Project 2: Healthcare Data Analysis
## Patient Outcomes & Hospital Performance

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
from utils.data_analysis_utils import utils
print("✅ Imports OK")


✅ Imports OK


In [ ]:
np.random.seed(42)
n = 8000
conditions = {
    'Hypertension': (3, 5000), 'Diabetes':    (4, 6000),
    'Pneumonia':    (5, 8000), 'Heart Disease':(7,15000),
    'Stroke':       (8,20000), 'Fracture':     (6,12000),
    'Cancer':       (10,30000),'COVID-19':     (9,25000),
}
departments = ['Cardiology','Pediatrics','Orthopedics','Neurology',
               'Oncology','Emergency','General Medicine','Psychiatry']

data = []
for i in range(n):
    cond, (base_los, base_cost) = list(conditions.items())[np.random.randint(0,8)]
    sev  = np.random.choice([1,2,3,4,5], p=[0.1,0.2,0.3,0.25,0.15])
    los  = max(1, round(np.random.normal(base_los*(1+(sev-3)*0.15),
                                         base_los*0.2), 1))
    cost = max(1000, int(np.random.normal(base_cost*(1+(sev-3)*0.25),
                                          base_cost*0.15)))
    data.append({
        'PatientID':          f'PT{str(i+1).zfill(6)}',
        'Age': int(np.clip(np.random.normal(52,18), 0, 100)),
        'Gender':             np.random.choice(['Male','Female']),
        'Department':         np.random.choice(departments),
        'PrimaryCondition':   cond,
        'Severity':           sev,
        'LengthOfStay':       los,
        'TreatmentCost':      cost,
        'Readmitted':         int(np.random.binomial(1, 0.05+(sev-1)*0.03)),
        'PatientSatisfaction':round(min(5, max(1, np.random.normal(4.2,0.8)-los/50)),1),
        'StaffToPatientRatio': round(float(np.clip(np.random.normal(4.5,0.8), 2, 8)), 1),
        'InsuranceType':      np.random.choice(['Private','Medicare','Medicaid','Self-Pay'],
                                               p=[0.4,0.3,0.2,0.1]),
        'Comorbidities':      np.random.randint(0,5),
        'EmergencyVisit':     np.random.choice([0,1], p=[0.7,0.3]),
    })

df_h = pd.DataFrame(data)
missing_idx = np.random.choice(df_h.index, 200, replace=False)
df_h.loc[missing_idx,'PatientSatisfaction'] = np.nan
df = utils.clean_dataframe(df_h, handle_missing='median')
print(f"✅ {len(df):,} patient records  |  shape={df.shape}")


AttributeError: 'float' object has no attribute 'clip'

In [ ]:
os.makedirs('visualizations', exist_ok=True)

dept = df.groupby('Department').agg(
    Avg_LOS=('LengthOfStay','mean'),
    Avg_Cost=('TreatmentCost','mean'),
    Satisfaction=('PatientSatisfaction','mean'),
    Readmission=('Readmitted','mean'),
    Patients=('PatientID','count')
).round(2)
dept['Readmission'] *= 100

fig, axes = plt.subplots(2, 2, figsize=(15,10))
dept.sort_values('Avg_LOS').plot.barh(y='Avg_LOS', ax=axes[0,0], color='steelblue', legend=False)
axes[0,0].set_title('Avg Length of Stay by Dept', fontweight='bold')
dept.sort_values('Avg_Cost').plot.barh(y='Avg_Cost', ax=axes[0,1], color='coral', legend=False)
axes[0,1].set_title('Avg Treatment Cost by Dept', fontweight='bold')
dept.sort_values('Satisfaction').plot.barh(y='Satisfaction', ax=axes[1,0], color='green', legend=False)
axes[1,0].set_title('Patient Satisfaction by Dept', fontweight='bold')
dept.sort_values('Readmission', ascending=False).plot.bar(y='Readmission', ax=axes[1,1], color='darkred', legend=False)
axes[1,1].set_title('Readmission Rate (%)', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/department_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: department_performance.png")


In [ ]:
# Age group analysis
bins   = [0,18,30,45,60,75,100]
labels = ['0-18','19-30','31-45','46-60','61-75','75+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
age = df.groupby('AgeGroup').agg(
    Avg_Cost=('TreatmentCost','mean'),
    Readmission=('Readmitted','mean')
).round(3)
age['Readmission'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14,6))
age['Avg_Cost'].plot.bar(ax=axes[0], color='teal')
axes[0].set_title('Treatment Cost by Age Group', fontweight='bold')
age['Readmission'].plot(ax=axes[1], marker='o', color='red')
axes[1].set_title('Readmission Rate by Age Group', fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/age_group_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: age_group_analysis.png")


In [ ]:
# Readmission prediction model
feats = ['Age','LengthOfStay','TreatmentCost','Severity',
         'Comorbidities','StaffToPatientRatio','EmergencyVisit']
X = df[feats].fillna(df[feats].mean())
y = df['Readmitted']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
acc = accuracy_score(y_te, y_pred)
auc = roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])
print(f"Accuracy: {acc*100:.1f}%  |  AUC: {auc:.3f}")
print(classification_report(y_te, y_pred, target_names=['Not Readmitted','Readmitted']))

fi = pd.Series(rf.feature_importances_, index=feats).sort_values()
fig, ax = plt.subplots(figsize=(10,6))
fi.plot.barh(ax=ax)
ax.set_title('Readmission Risk Factors', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/readmission_factors.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: readmission_factors.png")


In [ ]:
utils.plot_correlation_matrix(
    df[['Age','LengthOfStay','TreatmentCost','PatientSatisfaction',
        'StaffToPatientRatio','Comorbidities','Severity']],
    save_path='visualizations/correlation_matrix.png')

df.to_csv('processed_healthcare_data.csv', index=False)
dept.to_csv('department_performance.csv')
print("✅ Project 2 complete.")
